# `prep_TRGB_data` — build the TRGB-track SN dataset

This is the TRGB counterpart of [`prep_SN_data.ipynb`](prep_SN_data.ipynb). It assembles the
SN observation vector `y`, covariance submatrix `C`, and row labels used by
[`demo_TRGB_to_H0.ipynb`](demo_TRGB_to_H0.ipynb), and writes them to
`data/TRGB_{y,C,labels}.*` (the files shipped with the repo).

**Calibrators:** all SH0ES R22 calibrator hosts plus the three CATS-only hosts
NGC 1316, NGC 1404, NGC 4526 (Hoyt et al. 2023), matched to Pantheon+ by CID+IDSURVEY.
**Hubble-flow:** CATS-style selection from Pantheon+ (0.023 < z_HD < 0.15, IS_CALIBRATOR == 0).

> Requires `data/Pantheon+SH0ES_STAT+SYS.cov` (~32 MB, not in the repo — download from the
> [Pantheon+SH0ES release](https://github.com/PantheonPlusSH0ES/DataRelease)). The
> non-interactive equivalent is `data/build_TRGB_partial.py`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Pantheon+ data (standardized, bias-corrected m_B in 'm_b_corr')
df_pplus = pd.read_csv('data/Pantheon+SH0ES.dat', sep=r'\s+', comment='#')

# full STAT+SYS covariance matrix
cov_file = 'data/Pantheon+SH0ES_STAT+SYS.cov'
cov_data_flat = np.loadtxt(cov_file, skiprows=1)
cov_matrix = cov_data_flat.reshape((len(df_pplus), len(df_pplus)))

# TRGB calibrator label table (SH0ES R22 hosts + 3 CATS-only hosts)
df_trgb_labels = pd.read_csv('data/TRGB_extended_y_labels.csv')
print(f'Pantheon+: {len(df_pplus)} SNe;  TRGB calibrator rows: '
      f"{df_trgb_labels['calib_host'].notna().sum()}")

## Match calibrators to Pantheon+

`TRGB_extended_y_labels.csv` mixes two index systems in its `Unnamed: 0` column, so we match
every calibrator to Pantheon+ by the unambiguous `CID + IDSURVEY` key instead of by row index.

In [ ]:
df_pplus['key'] = df_pplus['CID'].astype(str) + '_' + df_pplus['IDSURVEY'].astype(str)
pp_key_to_idx = {k: i for i, k in enumerate(df_pplus['key'])}

cal_src = df_trgb_labels[df_trgb_labels['calib_host'].notna()]
calib_records, skipped = [], []
for _, row in cal_src.iterrows():
    host = str(row['calib_host']).lower()
    key  = f"{row['CID']}_{int(float(row['IDSURVEY']))}"
    if key not in pp_key_to_idx:
        skipped.append((host, key)); continue
    calib_records.append({'orig_index': pp_key_to_idx[key],
                          'label': f"{host}_{key}", 'type': 'CAL', 'host': host})
if skipped:
    print('Skipped (not in Pantheon+):', skipped)

calib_labels = pd.DataFrame(calib_records)
idx_calib = calib_labels['orig_index'].values
print(f"Calibrators: {len(calib_labels)} SN obs across {calib_labels['host'].nunique()} hosts")

## y vector and kinematic correction

In [ ]:
c = 299792.458  # speed of light in km/s

def q0_jerk_correction(z, q0, j0):
    return 0.5*(1 - q0)*z - (1/6)*(1 - q0 - 3*q0**2 + j0)*z**2

## Select Hubble-flow SNe (CATS-style)

Redshift 0.023 < z_HD < 0.15, `IS_CALIBRATOR == 0`, light-curve cuts (non-restrictive),
and exclude any SN that is one of our calibrators. Adjust the cuts below to explore.

In [ ]:
calib_cid_set = set(df_pplus.loc[idx_calib, 'CID'].astype(str).str.lower())

hf_mask = (
    df_pplus['zHD'].between(0.023, 0.15) &
    (df_pplus['IS_CALIBRATOR'] == 0) &
    df_pplus['x1'].between(-3, 3) &
    df_pplus['c'].between(-0.3, 0.3) &
    ~df_pplus['CID'].astype(str).str.lower().isin(calib_cid_set)
)
SNe_HF = df_pplus[hf_mask]
print(f'Hubble-flow SNe: {len(SNe_HF)}')

# calibrators: observed m_B; HF: subtract 5log10(cz{}) + 25 to get the Hubble intercept
y_calib = df_pplus.loc[idx_calib, 'm_b_corr'].values
z_HF    = SNe_HF['zHD'].values  # <-- try zCMB / zHEL
logcz   = 5*np.log10(c*z_HF*(1 + q0_jerk_correction(z_HF, q0=-0.55, j0=1)))  # <-- try other q0/j0
y_HF    = SNe_HF['m_b_corr'].values - logcz - 25

In [ ]:
# combine y vectors and labels
y   = np.concatenate([y_calib, y_HF])
idx = np.concatenate([idx_calib, SNe_HF.index.values]).astype(int)

hf_labels = pd.DataFrame({
    'orig_index': SNe_HF.index.values,
    'label': (SNe_HF['CID'].astype(str) + '_' + SNe_HF['IDSURVEY'].astype(str)).values,
    'type': 'HF', 'host': '',
})
labels = pd.concat([calib_labels, hf_labels], ignore_index=True).drop(columns='orig_index')
print(f"Calibrators: {(labels['type']=='CAL').sum()}, HF SNe: {(labels['type']=='HF').sum()}")
labels.head()

## Save outputs for `demo_TRGB_to_H0.ipynb`

Extracts the covariance submatrix for the selected rows and writes the three default files.
(Overwrites the shipped `data/TRGB_*`; rename the outputs below if you want to keep both.)

In [ ]:
C = cov_matrix[np.ix_(idx, idx)]

np.save('data/TRGB_y.npy', y)
np.save('data/TRGB_C.npy', C)
labels.to_csv('data/TRGB_labels.csv', index=False)

print(f"Saved {len(y)} rows.")
print(f"  TRGB_y.npy      : shape {y.shape}")
print(f"  TRGB_C.npy      : shape {C.shape}")
print(f"  TRGB_labels.csv : {len(labels)} rows")